# 🧬 Dark Manifold - REAL Luthey-Schulten CME_ODE Data

This notebook uses the **actual model data** from:
```
https://github.com/Luthey-Schulten-Lab/Minimal_Cell/tree/main/CME_ODE/model_data
```

## Key Files in model_data/
Based on their related repo (Minimal_Cell_ComplexFormation), this contains:
- `kinetic_params.xlsx` — Real Km, Vmax, kcat values
- `initial_concentration.xlsx` — Initial metabolite/protein concentrations
- Reaction stoichiometries for ODE solver
- Gene expression data

In [ ]:
#@title 1️⃣ Clone Repository & Check Structure
!git clone https://github.com/Luthey-Schulten-Lab/Minimal_Cell.git
print("\n" + "="*60)
print("CME_ODE/model_data contents:")
print("="*60)
!ls -la Minimal_Cell/CME_ODE/model_data/

In [ ]:
#@title 2️⃣ Explore All Files in model_data
import os

model_data_path = "Minimal_Cell/CME_ODE/model_data"

print(f"Contents of {model_data_path}:")
print("="*60)

for root, dirs, files in os.walk(model_data_path):
    level = root.replace(model_data_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        if size > 1_000_000:
            size_str = f"{size/1_000_000:.1f} MB"
        elif size > 1000:
            size_str = f"{size/1000:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"{subindent}📄 {file} ({size_str})")

In [ ]:
#@title 3️⃣ Load Kinetic Parameters (kinetic_params.xlsx)
import pandas as pd

# Try to load kinetic parameters
kinetic_files = [
    "Minimal_Cell/CME_ODE/model_data/kinetic_params.xlsx",
    "Minimal_Cell/CME_ODE/model_data/kinetic_parameters.xlsx",
]

kinetics_df = None
for f in kinetic_files:
    if os.path.exists(f):
        print(f"Found: {f}")
        # Load all sheets
        xlsx = pd.ExcelFile(f)
        print(f"Sheets: {xlsx.sheet_names}")
        
        for sheet in xlsx.sheet_names:
            df = pd.read_excel(f, sheet_name=sheet)
            print(f"\n--- Sheet: {sheet} ({len(df)} rows) ---")
            print(f"Columns: {list(df.columns)}")
            display(df.head(5))
        break

# Also check for CSV versions
import glob
csv_files = glob.glob("Minimal_Cell/CME_ODE/model_data/*.csv")
print(f"\nCSV files found: {len(csv_files)}")
for f in csv_files:
    print(f"  {f}")

In [ ]:
#@title 4️⃣ Load Initial Concentrations

conc_files = [
    "Minimal_Cell/CME_ODE/model_data/initial_concentration.xlsx",
    "Minimal_Cell/CME_ODE/model_data/initial_concentrations.xlsx",
]

for f in conc_files:
    if os.path.exists(f):
        print(f"Found: {f}")
        xlsx = pd.ExcelFile(f)
        print(f"Sheets: {xlsx.sheet_names}")
        
        for sheet in xlsx.sheet_names:
            df = pd.read_excel(f, sheet_name=sheet)
            print(f"\n--- Sheet: {sheet} ({len(df)} rows) ---")
            print(f"Columns: {list(df.columns)}")
            display(df.head(10))
        break

In [ ]:
#@title 5️⃣ Load ALL Excel/CSV Files

import glob

all_data = {}

# Find all data files
excel_files = glob.glob("Minimal_Cell/CME_ODE/model_data/**/*.xlsx", recursive=True)
csv_files = glob.glob("Minimal_Cell/CME_ODE/model_data/**/*.csv", recursive=True)
tsv_files = glob.glob("Minimal_Cell/CME_ODE/model_data/**/*.tsv", recursive=True)

print(f"Found: {len(excel_files)} Excel, {len(csv_files)} CSV, {len(tsv_files)} TSV files")
print("="*60)

# Load Excel files
for f in excel_files:
    try:
        xlsx = pd.ExcelFile(f)
        for sheet in xlsx.sheet_names:
            df = pd.read_excel(f, sheet_name=sheet)
            key = f"{os.path.basename(f)}:{sheet}"
            all_data[key] = df
            print(f"✓ {key}: {len(df)} rows x {len(df.columns)} cols")
    except Exception as e:
        print(f"✗ {f}: {e}")

# Load CSV files
for f in csv_files:
    try:
        df = pd.read_csv(f)
        key = os.path.basename(f)
        all_data[key] = df
        print(f"✓ {key}: {len(df)} rows x {len(df.columns)} cols")
    except Exception as e:
        print(f"✗ {f}: {e}")

print(f"\nTotal datasets loaded: {len(all_data)}")

In [ ]:
#@title 6️⃣ Parse Reaction Data (rxns_ODE.py)

# The reactions are defined in Python files
rxn_files = glob.glob("Minimal_Cell/CME_ODE/**/*rxn*.py", recursive=True) + \
            glob.glob("Minimal_Cell/CME_ODE/**/*reaction*.py", recursive=True)

print(f"Found {len(rxn_files)} reaction definition files:")
for f in rxn_files:
    print(f"  {f}")
    
# Try to read the main reaction file
for f in rxn_files:
    if 'rxns_ODE' in f or 'rxn_ODE' in f:
        print(f"\n{'='*60}")
        print(f"Contents of {f} (first 200 lines):")
        print("="*60)
        with open(f, 'r') as file:
            lines = file.readlines()[:200]
            for i, line in enumerate(lines):
                print(f"{i+1:4d}: {line}", end='')
        break

In [ ]:
#@title 7️⃣ Check for Genome/Gene Data (syn3A.gb)

# GenBank file with full genome
gb_files = glob.glob("Minimal_Cell/**/*.gb", recursive=True) + \
           glob.glob("Minimal_Cell/**/*.gbk", recursive=True)

print(f"Found {len(gb_files)} GenBank files:")
for f in gb_files:
    size = os.path.getsize(f)
    print(f"  {f}: {size:,} bytes")

# Try to parse with BioPython
try:
    !pip install biopython -q
    from Bio import SeqIO
    
    for gb_file in gb_files:
        print(f"\nParsing {gb_file}...")
        record = SeqIO.read(gb_file, "genbank")
        
        # Extract genes
        genes = []
        for feature in record.features:
            if feature.type == "gene":
                gene_name = feature.qualifiers.get("gene", ["unknown"])[0]
                locus_tag = feature.qualifiers.get("locus_tag", ["unknown"])[0]
                genes.append({"gene": gene_name, "locus_tag": locus_tag})
        
        print(f"  Genome length: {len(record.seq):,} bp")
        print(f"  Total features: {len(record.features)}")
        print(f"  Genes found: {len(genes)}")
        print(f"\n  First 20 genes:")
        for g in genes[:20]:
            print(f"    {g['locus_tag']}: {g['gene']}")
        
        break
except Exception as e:
    print(f"Could not parse GenBank: {e}")

In [ ]:
#@title 8️⃣ Extract Metabolites from Initial Concentrations

metabolites = []
initial_conc = {}

# Look for metabolite data in loaded datasets
for key, df in all_data.items():
    print(f"\nChecking {key}...")
    print(f"  Columns: {list(df.columns)[:5]}")
    
    # Look for metabolite-related columns
    for col in df.columns:
        col_lower = col.lower()
        if any(x in col_lower for x in ['metabolite', 'species', 'name', 'id']):
            print(f"  Found metabolite column: {col}")
            mets = df[col].dropna().tolist()
            if len(mets) > len(metabolites):
                metabolites = mets
                print(f"    → {len(mets)} metabolites")

print(f"\n{'='*60}")
print(f"Total metabolites found: {len(metabolites)}")
if metabolites:
    print(f"First 30: {metabolites[:30]}")

In [ ]:
#@title 9️⃣ Build Real Dataset
import torch
import numpy as np

class RealSyn3ADataset:
    """Real Syn3A data from Luthey-Schulten Lab."""
    
    def __init__(self):
        self.genes = []
        self.metabolites = []
        self.reactions = []
        self.kinetic_params = {}  # {rxn_id: {Km, Vmax, kcat}}
        self.initial_concentrations = {}  # {met_id: conc_mM}
        self.stoichiometry = None
        self.source = "Luthey-Schulten Lab CME_ODE/model_data"
    
    def summary(self):
        print(f"\n{'='*60}")
        print("REAL SYN3A DATASET")
        print(f"{'='*60}")
        print(f"Source: {self.source}")
        print(f"Genes: {len(self.genes)}")
        print(f"Metabolites: {len(self.metabolites)}")
        print(f"Reactions: {len(self.reactions)}")
        print(f"Kinetic params: {len(self.kinetic_params)}")
        print(f"Initial conc: {len(self.initial_concentrations)}")
        if self.stoichiometry is not None:
            print(f"Stoichiometry: {self.stoichiometry.shape}")

# Create dataset
dataset = RealSyn3ADataset()

# Populate from what we found
if metabolites:
    dataset.metabolites = metabolites

# Try to get genes from GenBank
try:
    from Bio import SeqIO
    for gb_file in gb_files:
        record = SeqIO.read(gb_file, "genbank")
        for feature in record.features:
            if feature.type == "CDS":
                gene = feature.qualifiers.get("gene", [None])[0]
                product = feature.qualifiers.get("product", ["unknown"])[0]
                if gene:
                    dataset.genes.append(gene)
        break
except:
    pass

dataset.summary()

In [ ]:
#@title 🔟 Save Extracted Data

# Save everything we found
save_data = {
    'genes': dataset.genes,
    'metabolites': dataset.metabolites,
    'reactions': dataset.reactions,
    'kinetic_params': dataset.kinetic_params,
    'initial_concentrations': dataset.initial_concentrations,
    'source': dataset.source,
    'all_dataframes': {k: v.to_dict() for k, v in all_data.items()},
}

import pickle
with open('syn3a_real_data.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print("Saved: syn3a_real_data.pkl")
print(f"  Genes: {len(save_data['genes'])}")
print(f"  Metabolites: {len(save_data['metabolites'])}")
print(f"  DataFrames: {len(save_data['all_dataframes'])}")

# Download
from google.colab import files
files.download('syn3a_real_data.pkl')

## What This Notebook Extracts

From `CME_ODE/model_data/`:

| File | Contains |
|------|----------|
| `kinetic_params.xlsx` | Real Km, Vmax, kcat for all enzymes |
| `initial_concentration.xlsx` | Starting metabolite concentrations |
| `syn3A.gb` | Full genome with 493 genes |
| `rxns_ODE.py` | Reaction definitions with stoichiometry |

Run this first to see what's available, then we can train the model!